In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines

# Configure the plotting routines

import pandas as pd

# Import the CRUD module created in Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################


username = "aacuser"
password = "AacUser2026!"

# Connect to MongoDB through the CRUD Module
db = AnimalShelter(username, password)

# Retrieve the complete unfiltered dataset
records = db.read({})

# Convert MongoDB records to a Dataframe
df = pd.DataFrame.from_records(records)

# Remove MongoDB's ObjectId field
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)

# Verify that the data loaded correctly
print("shape:", df.shape)
print("columns:", df.columns.tolist())


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare logo
image_filename = '/home/codio/workspace/code_files/Grazioso Salvare Logo.png'

with open(image_filename, 'rb') as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode()

#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    
    # Dashboard title
    html.Center(
        html.B(
            html.H1('CS-340 Dashboard')
        )
    ),
    
    html.Hr(),
    
    # Grazioso Salvare logo
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={
                'width': '300px',
                'display': 'block',
                'margin': 'auto'
            }
        )
    ),
    
    # Dashboard title and unique identifier
    html.H3(
        'Grazioso Salvare Animal Rescue Dashboard',
        style={
            'textAlign': 'center'
        }
   
    ),
    
    html.H4(
        'Tajudeen Tijani',
        style={
            'textAlign': 'center'
        }
    ),
    
    html.Hr(),
        
    # Rescue type filter
    html.Div([
        html.Label(
            'Select Rescue Type:',
            style={
                'fontweight': 'bold',
                'fontSize': '18px'
            }
        ),
        dcc.Dropdown(
            id='filter-type',
            options=[
                {
                    'label': 'Water Rescue',
                    'value': 'water'
                },
                {
                    'label': 'Mountain or Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster or Individual Tracking',
                    'value': 'disaster'
                },
                {
                    'label': 'Reset',
                    'value': 'reset'
                }
            ],
            value='reset',
            clearable=False
        )
    ]),
    
    html.Hr(),
    
    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        
        columns=[
            {
                'name': i,
                'id': i,
                'deletable': False,
                'selectable': True
            }
            for i in df.columns
        ],
        
        data=df.to_dict('records'),
        
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        
        style_table={
            'overflowX': 'auto'
        },
        
        style_cell={
            'textAlign': 'left',
            'padding': '8px'
        },
        
        style_header={
            'fontWeight': 'bold'
        }
    ),
        
    html.Br(),
    html.Hr(),
    # Charts
    html.Div(
        className='row',
        style={
            'display': 'flex',
            'flexWrap': 'wrap'
        },
        
        children=[
            
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={
                    'width': '50%',
                    'height': '500px'
                }
            ),
            
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={
                    'width': '50%',
                    'height': '500px'
                }
            )
        ]
    )
])

    
@app.callback(
    Output('datatable-id','data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    
    # Reset returns all records
    if filter_type == 'reset':
        query = {}
    
    # Water Rescue   
    elif filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }
    
    # Mountain or Wilderness Rescue
    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }
        
    # Disaster or Indiidual Tracking
    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Rottweiler'
             ]
        }
    }
    
    else:
        query = {}
    
    # Use the Project One CRUD module to retrieve filtered data
    filtered_records = db.read(query)
    
    filtered_df = pd.DataFrame.from_records(filtered_records)
    
    # Remove MongoDB ObjectId before sending data to Dash
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
    return filtered_df.to_dict('records')

@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):
    
    if viewData is None:
        return []
    
    chart_df = pd.DataFrame.from_records(viewData)
    
    if chart_df.empty:
        return []
    
    breed_counts = chart_df['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Animal Breeds'
    )
    
    return [dcc.Graph(figure=fig)]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    
    if not selected_columns:
        return []
    
    return [
        {
            'if': {
                'column_id': column
            },
            'background_color': '#D2F3FF'
    } 
    for column in selected_columns]
    


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index): 
        
    if viewData is None:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
        
    if dff.empty:
        return []
        
    # If no row is selected, display the first row
    if not index:
        row = 0
    else:
        row = index[0]
    # Make sure the selected row is valid
    if row >= len(dff):
        row = 0
        
    return [
       dl.Map(
            style={
                'width': '1000px', 
                'height': '500px'
            }, 
            center=[30.75,-97.48], 
            zoom=10, 
            children=[
            dl.TileLayer(id='base-layer-id'),
            
            dl.Marker(
                position=[
                    dff.iloc[row,13],
                    dff.iloc[row,14]
                ], 
                children=[
                    dl.Tooltip(
                        dff.iloc[row,4]
                    ),
                    dl.Popup(
                        children=[
                            html.H4('Animal Name'),
                            html.P(
                                dff.iloc[row,9]
                            )
                        ]
                    )
                ]
            )
        ]
    )
]
        
                    
        
        
    


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

shape: (10002, 16)
columns: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']
Dash app running on https://sonarzoom-rivalbeyond-3000.codio.io/proxy/8050/
